### Hugging Face
- 이미지캡셔닝   :  Salesforce/blip-image-captioning-base
- 텍스트 임베딩  :  sentence-transformers/all-MiniLM-L6-v2
- 텍스트 생성    : distilbert/distilgpt2  (경량 모델)

In [1]:
import os
import fitz
import chromadb
from PIL import Image
from transformers import pipeline,BlipProcessor, BlipForConditionalGeneration
from chromadb.utils import embedding_functions

In [2]:
# BLIP 모델 로드
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [3]:
# 임베딩 모델 로드
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# vectorDB 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db_hf')
collection = chroma_client.get_or_create_collection(name='multimocal_rag_hf',embedding_function=hf_ef)

### 데이터추출 및 오픈소스 임베딩
- PDF 텍스트와 추출된 이미지를 BLIP 모델로 갭셔닝한 뒤, Sentence Transformers임베딩을 사용해서 로컬 vectorDB에 저장

In [ ]:
ptf_path = 'sample_paper.pdf'
if collection.count() == 0:
    doc = fitz.open(ptf_path)
    documents,metadatas,ids = [],[],[]
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추가
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metadatas.append({'type':'text','page':page_num+1})
            ids.append(f'hf_text_page_{page_num+1}')
        # 이미지 캡셔닝
        image_lists = page.get_images(full=True)
        if image_lists:
            xref = image_lists[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            image_filename = f'ht_temp_img.{ext}'
            with open(image_filename, 'wb') as f:
                f.write(image_bytes)
            
            try:
                raw_image = Image.open(image_filename).convert('RGB')
                inputs = processor(raw_image, return_tensors='pt')
                out = model.generate(**inputs,max_new_tokens=50)
                caption = processor.decode(out[0],skip_special_tokens=True)

                documents.append(caption)
                metadatas.append({'type':'image_summary', 'page':page_num+1})
                ids.append(f'hf_image_summary_page_{page_num+1}')
                print(f'페이지 {page_num+1} 이미지 캡셔닝 완료 : {caption}')
            except Exception as e:
                print(f'페이지 {page_num+1} 이미지 캡셔닝 실패 : {e}')
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
else:
    print(collection.count())


[transformers] Keyword argument `return_tensor` is not a valid argument for this processor and will be ignored.


페이지 3 이미지 캡셔닝 실패 : 'list' object has no attribute 'shape'
페이지 4 이미지 캡셔닝 실패 : 'list' object has no attribute 'shape'
